# 00 · Prerequisites Bootcamp: Excel → Python for demand planners

**Business question:** *Can I do in Python everything I already do in Excel with demand data, and then more?*

This notebook teaches only what the project needs, using the **real Intel microprocessor data** we'll analyse later. Every step shows the Excel equivalent.

**How to use a notebook**
- Click a grey code cell and press **Shift + Enter** to run it. The result appears below.
- Run cells **top to bottom**. If things get weird, use *Restart* → *Run All*.
- Top-right kernel should say **Semiconductor DP (.venv)**.

**Sections:** 1. Python basics · 2. Load data · 3. Filter & sort · 4. Formula columns · 5. PivotTables & SUMIFS · 6. XLOOKUP · 7. Moving averages & time shifts · 8. Charts · 9. Back to Excel · 10. Git basics · 11. Exercises

## 1. Python basics (just enough)
A **variable** is like a named cell. A **list** is a column of values. A **dict** is a little lookup table (key → value). A **function** is a reusable formula you write once.

In [ ]:
lead_time_weeks = 4                      # a variable (like cell B1 = 4)
weekly_orders = [520, 480, 610, 550]     # a list (like a column A1:A4)
asp_by_group = {1: "High", 2: "Mid", 3: "Low"}   # a dict (like a 2-column lookup table)

def average(values):
    """Like =AVERAGE(A1:A4)"""
    return sum(values) / len(values)

avg = average(weekly_orders)
print(f"Average weekly orders: {avg:.0f} units")          # f-string = text with values inserted
print(f"Demand over the lead time: {avg * lead_time_weeks:.0f} units")
print("ASP group 2 means:", asp_by_group[2])

for week, qty in enumerate(weekly_orders, start=1):      # a loop = 'for each row'
    flag = "HIGH" if qty > avg else "normal"             # like =IF(A1>avg,"HIGH","normal")
    print(f"Week {week}: {qty} -> {flag}")

## 2. Load the data (= File → Open)
`pandas` is Python's spreadsheet engine. A **DataFrame** (`df`) is a sheet: rows and named columns.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path.cwd().parent / "data" / "raw" / "intel_msom_2022.csv"   # notebooks/ is one level down
df = pd.read_csv(DATA)

print("Rows x columns:", df.shape)
df.head()          # first 5 rows, like scrolling to the top of the sheet

In [ ]:
df.info()          # column names + data types (like checking cell formats)

In [ ]:
df.describe().round(1)   # quick stats per numeric column: count, mean, std, min, quartiles, max

**Reading the columns**
- `Distribution Center`: one of Intel's 5 warehouses (Greek-letter names).
- `Product Offering`: product line letter.
- `Generation`: product generation (newer = higher).
- `SKU`: product line + generation, e.g. `SKU-A-2`.
- `ASP Group`: average selling price category.
- `Week`: 1 … 187 (relative weeks, not calendar dates).
- `Forecasted Demand`: Intel's forecast for that week.
- `Customer Orders`: what customers actually ordered.

## 3. Filter & sort (= AutoFilter and Sort)

In [ ]:
# Excel: AutoFilter SKU = 'SKU-A-2' AND DC = 'ALPHA', then sort by Week
one = df[(df["SKU"] == "SKU-A-2") & (df["Distribution Center"] == "ALPHA")].sort_values("Week")
print(len(one), "weeks of data for SKU-A-2 at ALPHA")
one.head(8)

In [ ]:
# Unique values (= the filter drop-down list)
print("DCs:", sorted(df["Distribution Center"].unique()))
print("Number of SKUs:", df["SKU"].nunique())
print("Generations:", sorted(df["Generation"].unique()))

## 4. Formula columns (= typing a formula in a new column and filling down)
In pandas, one line applies the formula to **every row at once**. No fill-down needed.

In [ ]:
df["Error"] = df["Forecasted Demand"] - df["Customer Orders"]     # + means over-forecast
df["Abs Error"] = df["Error"].abs()                                # =ABS(...)
df[["SKU", "Week", "Forecasted Demand", "Customer Orders", "Error", "Abs Error"]].head()

In [ ]:
# The project's main KPI, WMAPE = SUMPRODUCT(ABS(F-A)) / SUM(A)
wmape = df["Abs Error"].sum() / df["Customer Orders"].sum()
bias_pct = df["Error"].sum() / df["Customer Orders"].sum()
print(f"Intel forecast, all data: WMAPE = {wmape:.1%}  ->  accuracy = {1 - wmape:.1%},  bias = {bias_pct:+.1%}")
print("(A first peek only. Day 2 does this properly, by lag, SKU and DC, with significance tests.)")

## 5. PivotTables & SUMIFS (= `groupby` and `pivot_table`)

In [ ]:
# SUMIFS-style subtotal: total orders per DC
df.groupby("Distribution Center")["Customer Orders"].sum().sort_values(ascending=False)

In [ ]:
# A real PivotTable: rows = Generation, columns = DC, values = sum of orders
pd.pivot_table(df, index="Generation", columns="Distribution Center",
               values="Customer Orders", aggfunc="sum", margins=True, margins_name="Total")

In [ ]:
# Several KPIs per group at once (like a PivotTable with multiple value fields)
by_dc = df.groupby("Distribution Center").agg(
    orders=("Customer Orders", "sum"),
    forecast=("Forecasted Demand", "sum"),
    abs_error=("Abs Error", "sum"),
)
by_dc["wmape"] = by_dc["abs_error"] / by_dc["orders"]
by_dc["bias_%"] = (by_dc["forecast"] - by_dc["orders"]) / by_dc["orders"]
by_dc.style.format({"orders": "{:,.0f}", "forecast": "{:,.0f}", "abs_error": "{:,.0f}",
                    "wmape": "{:.1%}", "bias_%": "{:+.1%}"})

## 6. XLOOKUP (= `merge`)
Bring information from another table into this one by matching a key.

In [ ]:
# A small lookup table: ASP group -> label (in real life this might be a product master sheet)
asp_lookup = pd.DataFrame({"ASP Group": sorted(df["ASP Group"].unique())})
asp_lookup["Price Tier"] = ["Tier " + str(g) for g in asp_lookup["ASP Group"]]

df = df.merge(asp_lookup, on="ASP Group", how="left")   # = XLOOKUP(ASP Group, lookup[ASP Group], lookup[Price Tier])
df[["SKU", "ASP Group", "Price Tier"]].drop_duplicates().head()

## 7. Moving averages & time shifts (= helper columns referencing rows above)
- `.shift(1)` = "the value one row above" (last week).
- `.rolling(4).mean()` = a 4-week moving average.

⚠️ We use `.shift(1)` **before** rolling, so the average uses only *past* weeks. Including the current week would be **data leakage**: using the answer to predict the answer.

In [ ]:
one = one.copy()
one["Last Week"] = one["Customer Orders"].shift(1)
one["MA4 (past 4 wks)"] = one["Customer Orders"].shift(1).rolling(4).mean()   # a simple moving-average forecast
one[["Week", "Customer Orders", "Last Week", "MA4 (past 4 wks)", "Forecasted Demand"]].head(8)

In [ ]:
# Which forecast was better for this SKU-DC: Intel's, or a simple 4-week moving average?
valid = one.dropna(subset=["MA4 (past 4 wks)"])
for name, col in [("Intel forecast", "Forecasted Demand"), ("4-week moving average", "MA4 (past 4 wks)")]:
    w = (valid[col] - valid["Customer Orders"]).abs().sum() / valid["Customer Orders"].sum()
    print(f"{name:<24} WMAPE = {w:.1%}")
print("\nThis is a tiny preview of 'Forecast Value Add' - comparing forecasts fairly on the same weeks.")

## 8. Charts (= Insert → Line chart)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(one["Week"], one["Customer Orders"], label="Customer orders (actual)", linewidth=1.8)
ax.plot(one["Week"], one["Forecasted Demand"], label="Intel forecast", linewidth=1.2, alpha=0.8)
ax.plot(one["Week"], one["MA4 (past 4 wks)"], label="4-week moving average", linewidth=1.2, linestyle="--")
ax.set(title="SKU-A-2 at ALPHA: orders vs. forecasts", xlabel="Week", ylabel="Units")
ax.legend()
plt.show()

## 9. Back to Excel (= Save As .xlsx)
Planners and managers live in Excel. A Python model is only useful if its results reach them.

In [ ]:
OUT = Path.cwd().parent / "data" / "processed" / "bootcamp_dc_summary.xlsx"
by_dc.to_excel(OUT, sheet_name="DC summary")
print("Saved:", OUT)

## 10. Git basics: how your work gets saved to GitHub
Think of Git as **"Track Changes + Version History" for a whole folder**, and GitHub as the cloud copy.

| Step | Command (PowerShell terminal in VS Code) | Meaning |
|---|---|---|
| 1 | `git status` | What changed since the last save point? |
| 2 | `git add notebooks/00_prerequisites_bootcamp.ipynb` | Put this file in the "ready to save" box (`git add -A` = everything) |
| 3 | `git commit -m "Complete bootcamp exercises"` | Create a save point with a message |
| 4 | `git push` | Upload save points to GitHub |
| 5 | `git log --oneline` | List of save points (your project's history) |

A good commit message says **what** changed, in the imperative mood: *"Add bias significance tests"*, not *"stuff"*.

In this project `/checkpoint` does steps 1–4 for you at the end of each phase. You'll still do it once by hand after the exercises below, so you know what's happening.

## 11. Exercises (5)
Try each one in the empty cell, **then** reveal the solution cell below it (click the "…" / expand arrow).

**Exercise 1.** How many rows belong to the `GAMMA` distribution center? *(Hint: filter, then `len()`.)*

In [ ]:
# your answer here

In [ ]:
print(len(df[df["Distribution Center"] == "GAMMA"]))

**Exercise 2.** Which **SKU** has the highest total customer orders across all DCs and weeks? *(Hint: `groupby` + `sum` + `idxmax()`.)*

In [ ]:
# your answer here

In [ ]:
totals = df.groupby("SKU")["Customer Orders"].sum()
print(totals.idxmax(), f"{totals.max():,.0f} units")

**Exercise 3.** Build a pivot table: rows = `Generation`, values = WMAPE of Intel's forecast. Which generation is hardest to forecast?

In [ ]:
# your answer here

In [ ]:
g = df.groupby("Generation")[["Abs Error", "Customer Orders"]].sum()
g["WMAPE"] = g["Abs Error"] / g["Customer Orders"]
print(g["WMAPE"].map("{:.1%}".format))
print("Hardest:", g["WMAPE"].idxmax())

**Exercise 4.** For SKU-A-2 at ALPHA, add a column `MA8` = 8-week moving average of *past* orders. Is its WMAPE better or worse than MA4?

In [ ]:
# your answer here

In [ ]:
one["MA8"] = one["Customer Orders"].shift(1).rolling(8).mean()
v = one.dropna(subset=["MA8"])
for col in ["MA4 (past 4 wks)", "MA8"]:
    print(col, f"{(v[col] - v['Customer Orders']).abs().sum() / v['Customer Orders'].sum():.1%}")
print("Longer windows are smoother but react more slowly to trends.")

**Exercise 5.** Count how many SKU-DC combinations exist, and how many weeks each has on average. *(Hint: `groupby([...]).size()`.)* Why might some have fewer than 187 weeks?

In [ ]:
# your answer here

In [ ]:
sizes = df.groupby(["SKU", "Distribution Center"]).size()
print(len(sizes), "SKU-DC combinations; average", round(sizes.mean(), 1), "weeks each")
print("Fewer than 187 weeks -> the product launched after week 1 or reached end-of-life before week 187 (life cycles!).")

---
## What a demand planner takes away
- Everything you do in Excel (filter, formulas, pivots, lookups, charts) is a line or two of pandas. It is repeatable, and it works on millions of rows.
- The project's core KPIs (WMAPE, bias) are just formula columns plus a `groupby`.
- **Using only past data** (`.shift(1)`) is the habit that separates real forecasting from fake accuracy.
- Results go back to Excel in one line, so you can always hand them to Excel-based teams.

**Checkpoint questions**
1. What is the pandas equivalent of a PivotTable, and of XLOOKUP?
2. Why do we write `.shift(1).rolling(4).mean()` instead of `.rolling(4).mean()` for a forecast?
3. In one sentence, what's the difference between `git commit` and `git push`?